# Visit with Us: Wellness Tourism Package Prediction

## End-to-end MLOps pipeline with GitHub Actions and Streamlit

### Business objective

Visit with Us is launching a Wellness Tourism Package and needs a consistent way to decide which customers should be contacted first. This project builds a reproducible machine-learning workflow that estimates a customer's purchase likelihood before outreach. The workflow validates the data, prepares a train/test split, tunes and evaluates a model, records experiment evidence, and exposes the approved model through a Streamlit interface.

**Target:** `ProdTaken` - `1` means the customer purchased a package; `0` means they did not.

> This notebook is intentionally paired with a clean GitHub repository. The notebook demonstrates the executed analysis and the repository contains the reusable scripts, CI/CD workflow, model, and Streamlit app.


## Rubric coverage map

| Rubric area | Evidence in this submission |
|---|---|
| Data registration | `data/tourism.csv`, `src/data_registration.py`, and the executed validation summary below |
| Data preparation | `src/data_preparation.py`, reproducible 80:20 stratified split, and `train-test-splits` GitHub artifact |
| Model building and tracking | Tuned Random Forest, 5-fold `GridSearchCV`, `experiments/grid_search_results.csv`, metrics, and MLflow-compatible logging |
| Deployment | `app.py`, the committed `.joblib` model, and `requirements.txt` |
| CI/CD | `.github/workflows/pipeline.yml` with register, preparation, and training jobs |
| Output evaluation | A final evidence section for public GitHub and Streamlit URLs/screenshots |


## 1. Project setup and reproducibility

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd() / "visit_with_us_mlops"
if not PROJECT_ROOT.exists():
    # In the GitHub repository / Colab, set this to the cloned repository folder.
    PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "tourism.csv"
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data path: {DATA_PATH}")
print(f"Dataset exists: {DATA_PATH.exists()}")


Project root: /workspace/scratch/36314d7eb4f9/visit_with_us_mlops
Raw data path: /workspace/scratch/36314d7eb4f9/visit_with_us_mlops/data/tourism.csv
Dataset exists: True


The repository uses fixed package versions in `requirements.txt` and a fixed `random_state=42`. This makes the split and model-selection result repeatable across runs.

## 2. Data registration

The raw file is stored inside the repository under `data/tourism.csv`. The registration component verifies the required business columns, rejects non-binary target values, checks that the file has rows, and writes a JSON summary for the workflow.


In [2]:
from src.data_registration import register_dataset

registration_summary = register_dataset(
    DATA_PATH,
    PROJECT_ROOT / "reports" / "data_registration_summary.json",
)
print("DATA REGISTRATION: PASSED")
print(f"Rows: {registration_summary['rows']:,} | Columns: {registration_summary['columns']}")
print(f"Duplicate rows: {registration_summary['duplicate_rows']}")
print(f"Target distribution: {registration_summary['target_distribution']}")


DATA REGISTRATION: PASSED
Rows: 4,128 | Columns: 21
Duplicate rows: 0
Target distribution: {'0': 3331, '1': 797}


In [3]:
raw_df = pd.read_csv(DATA_PATH)
print(raw_df.shape)
print(raw_df.head(5).to_string(index=False))


(4128, 21)
 Unnamed: 0  CustomerID  ProdTaken  Age   TypeofContact  CityTier  DurationOfPitch  Occupation Gender  NumberOfPersonVisiting  NumberOfFollowups ProductPitched  PreferredPropertyStar MaritalStatus  NumberOfTrips  Passport  PitchSatisfactionScore  OwnCar  NumberOfChildrenVisiting Designation  MonthlyIncome
          0      200000          1 41.0    Self Enquiry         3              6.0    Salaried Female                       3                3.0         Deluxe                    3.0        Single            1.0         1                       2       1                       0.0     Manager        20993.0
          1      200001          0 49.0 Company Invited         1             14.0    Salaried   Male                       3                4.0         Deluxe                    4.0      Divorced            2.0         0                       3       1                       2.0     Manager        20130.0
          2      200002          1 37.0    Self Enquiry         1   

### Data-quality findings

| Column | Missing values |
| --- | --- |
| Unnamed: 0 | 0 |
| CustomerID | 0 |
| ProdTaken | 0 |
| Age | 0 |
| TypeofContact | 0 |
| CityTier | 0 |
| DurationOfPitch | 0 |
| Occupation | 0 |
| Gender | 0 |
| NumberOfPersonVisiting | 0 |
| NumberOfFollowups | 0 |
| ProductPitched | 0 |
| PreferredPropertyStar | 0 |
| MaritalStatus | 0 |
| NumberOfTrips | 0 |
| Passport | 0 |
| PitchSatisfactionScore | 0 |
| OwnCar | 0 |
| NumberOfChildrenVisiting | 0 |
| Designation | 0 |
| MonthlyIncome | 0 |

The provided CSV has **no missing values**. It does have an accidental index column (`Unnamed: 0`) and a customer identifier (`CustomerID`), neither of which should be learned by the model. The model pipeline still contains imputers so that a future batch with missing values can be served safely.

| ProdTaken | Customers | Meaning | Share |
| --- | --- | --- | --- |
| 0 | 3331 | Did not purchase | 80.7% |
| 1 | 797 | Purchased | 19.3% |

![Target distribution](notebook_assets/target_distribution.png)

**Observation:** only **19.3%** of customers purchased the package. Accuracy alone would be misleading, so model selection uses ROC-AUC and the report also includes precision, recall, F1, and average precision.


## 3. Data preparation

The preparation step cleans category-label inconsistencies, removes the two non-predictive ID columns, and writes an 80:20 stratified split. Stratification preserves the purchase rate in both datasets. The GitHub workflow uploads the four output CSVs as a `train-test-splits` artifact, which the model-training job downloads.


In [4]:
from src.data_preparation import prepare_splits

prep_summary = prepare_splits(DATA_PATH, PROJECT_ROOT / "data" / "processed")
print("DATA PREPARATION: PASSED")
print(f"Dropped columns: {prep_summary['dropped_columns']}")
print(f"Train rows: {prep_summary['training_rows']:,} | Test rows: {prep_summary['test_rows']:,}")
print(f"Train purchase rate: {prep_summary['train_positive_rate']:.2%}")
print(f"Test purchase rate: {prep_summary['test_positive_rate']:.2%}")


DATA PREPARATION: PASSED
Dropped columns: ['Unnamed: 0', 'CustomerID']
Train rows: 3,302 | Test rows: 826
Train purchase rate: 19.32%
Test purchase rate: 19.25%


**Cleaning decisions:**

- Removed `Unnamed: 0` because it is a CSV export index, not customer behaviour.
- Removed `CustomerID` because it is a unique identifier and would not generalize to unseen customers.
- Standardized `Fe Male` to `Female` and `Unmarried` to `Single` to prevent data-entry variants from becoming separate model categories.
- Did not remove different customers with identical profiles; those are legitimate distinct observations after identifiers are excluded.


## 4. Exploratory observations

| ProductPitched | Customers | Purchase rate |
| --- | --- | --- |
| Basic | 1615 | 30.0% |
| Standard | 737 | 16.1% |
| Deluxe | 1422 | 11.6% |
| King | 104 | 8.7% |
| Super Deluxe | 250 | 7.6% |

![Purchase rate by product pitched](notebook_assets/purchase_rate_by_product.png)

**Observation:** `Basic` has the highest observed purchase rate in this historical data (30.0%). This is an association, not a causal claim; campaign decisions should use the full model score rather than a single field. Passport holders also have a markedly higher raw conversion rate (35.8%) than non-holders (12.4%).


## 5. Model building and experimentation tracking

### Modelling approach

A `ColumnTransformer` handles numerical and categorical values separately, and it is bundled with a class-balanced `RandomForestClassifier` in one scikit-learn pipeline. Keeping preprocessing and the estimator together prevents training/serving drift.

The pipeline tunes these parameters through five-fold `GridSearchCV` using **ROC-AUC** as the selection metric:

- `n_estimators`: 200 or 350
- `max_depth`: unrestricted or 12
- `min_samples_leaf`: 1 or 3
- `max_features`: `sqrt`

Every candidate and its cross-validation score is written to `experiments/grid_search_results.csv`. In GitHub Actions, MLflow is installed from `requirements.txt` and records parameters, metrics, the grid-search file, and the approved model. A file-based experiment log is retained as a safe local fallback.


In [5]:
from src.train_model import train_and_evaluate

training_result = train_and_evaluate(PROJECT_ROOT / "data" / "processed")
metrics = training_result["metrics"]
print("MODEL TRAINING: PASSED")
print(f"Best CV ROC-AUC: {metrics['best_cv_roc_auc']:.4f}")
print(f"Test ROC-AUC: {metrics['roc_auc']:.4f} | Test F1: {metrics['f1_score']:.4f}")
print(f"Best parameters: {training_result['metadata']['best_parameters']}")
print(f"Saved model: {training_result['metadata']['model_path']}")


MODEL TRAINING: PASSED
Best CV ROC-AUC: 0.9541
Test ROC-AUC: 0.9747 | Test F1: 0.7188
Best parameters: {'classifier__max_depth': None, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__n_estimators': 350}
Saved model: /workspace/scratch/36314d7eb4f9/visit_with_us_mlops/models/wellness_package_model.joblib


In [6]:
grid_results = pd.read_csv(PROJECT_ROOT / "experiments" / "grid_search_results.csv")
print(grid_results.sort_values("rank_test_score").to_string(index=False))


                                                                                                                                 params mean_test_score std_test_score mean_train_score  rank_test_score
{'classifier__max_depth': None, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__n_estimators': 350}          0.9541         0.0073           1.0000                1
{'classifier__max_depth': None, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__n_estimators': 200}          0.9540         0.0071           1.0000                2
  {'classifier__max_depth': 12, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__n_estimators': 350}          0.9438         0.0065           1.0000                3
  {'classifier__max_depth': 12, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__n_estimators': 200}          0.9434         0.0086           1.0000             

### Hold-out test evaluation

| Metric | Value | Why it matters |
| --- | --- | --- |
| Cross-validation ROC-AUC | 0.9541 | Model-selection metric across five folds |
| Test ROC-AUC | 0.9747 | Ability to rank purchasers above non-purchasers |
| Test average precision | 0.9150 | Precision-recall quality for the imbalanced target |
| Accuracy | 0.9128 | Overall classification correctness |
| Precision | 0.9485 | Share of contacted high-priority leads who purchased |
| Recall | 0.5786 | Share of purchasers identified at the 0.50 threshold |
| F1 score | 0.7188 | Balance of precision and recall |

![Confusion matrix](reports/confusion_matrix.png)

At the default 0.50 probability threshold, the model identified **92 of 159 actual purchasers** and generated only **5 false-positive leads**. This high precision is useful when sales follow-up capacity is expensive. If the business wants to find more potential purchasers, the team can lower the probability threshold after agreeing on an acceptable increase in false positives.

### Feature signals

| feature | importance |
| --- | --- |
| numeric__Age | 0.1074 |
| numeric__MonthlyIncome | 0.1034 |
| numeric__DurationOfPitch | 0.1011 |
| numeric__Passport | 0.0928 |
| numeric__NumberOfTrips | 0.0568 |
| numeric__PitchSatisfactionScore | 0.0561 |
| numeric__NumberOfFollowups | 0.0473 |
| numeric__PreferredPropertyStar | 0.0418 |
| numeric__CityTier | 0.0405 |
| categorical__MaritalStatus_Single | 0.0337 |

![Feature importance](reports/feature_importance.png)

The strongest signals are age, monthly income, pitch duration, passport ownership, and travel behaviour. Feature importance describes how the model used this historical data; it does not prove that changing a customer characteristic will cause a purchase.


## 6. CI/CD workflow with GitHub Actions

The project includes `.github/workflows/pipeline.yml`. The workflow triggers on a push to `main` that changes data, source code, tests, requirements, or the workflow itself. It can also be started manually through `workflow_dispatch`.

| GitHub Actions job | Main actions | Handoff/output |
|---|---|---|
| `register-data` | Installs dependencies, executes a schema test, runs registration | `registered-dataset` artifact plus validation summary |
| `prepare-data` | Downloads the registered data, cleans it, creates the stratified split | `train-test-splits` artifact |
| `train-model` | Downloads splits, tunes/tracks/evaluates the model, runs quality gate | Saved model, reports, experiments, and an approved commit to `main` |

The quality gate requires test ROC-AUC of at least 0.80 and recall of at least 0.50 before the bot commits refreshed model artifacts. This prevents a failed run or weak replacement model from silently reaching the deployed app.


In [7]:
print((PROJECT_ROOT / ".github" / "workflows" / "pipeline.yml").read_text())


name: Wellness Tourism MLOps Pipeline

on:
  push:
    branches: [main]
    paths:
      - "data/tourism.csv"
      - "src/**"
      - "tests/**"
      - "requirements.txt"
      - ".github/workflows/pipeline.yml"
  workflow_dispatch:

permissions:
  contents: write

jobs:
  register-data:
    name: Register and validate dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - name: Install project dependencies
        run: python -m pip install --upgrade pip && python -m pip install -r requirements.txt
      - name: Run unit test for data registration
        run: python -m unittest discover -s tests -v
      - name: Register raw dataset
        run: python src/data_registration.py --input data/tourism.csv
      - name: Upload registered dataset and validation report
        uses: actions/upload-artifact@v4
        with:
          name: registered-dataset
          path:

## 7. Model deployment with Streamlit

`app.py` loads `models/wellness_package_model.joblib`, presents customer and interaction input controls, stores the selections in a one-row pandas DataFrame, and returns a purchase probability and prioritization message.

The application uses exactly the features used during training, while the saved scikit-learn pipeline applies the same preprocessing during inference. `requirements.txt` provides the Streamlit deployment dependencies; `deployment/requirements.txt` is also included as a deployment-only dependency reference.


In [8]:
import joblib

model = joblib.load(PROJECT_ROOT / "models" / "wellness_package_model.joblib")
sample_input = pd.DataFrame([{
    "Age": 35.0, "TypeofContact": "Self Enquiry", "CityTier": 2,
    "DurationOfPitch": 12.0, "Occupation": "Salaried", "Gender": "Female",
    "NumberOfPersonVisiting": 2, "NumberOfFollowups": 3.0,
    "ProductPitched": "Basic", "PreferredPropertyStar": 4.0,
    "MaritalStatus": "Single", "NumberOfTrips": 2.0, "Passport": 1,
    "PitchSatisfactionScore": 3, "OwnCar": 1,
    "NumberOfChildrenVisiting": 0.0, "Designation": "Executive",
    "MonthlyIncome": 25000.0,
}])
probability = model.predict_proba(sample_input)[:, 1][0]
print(sample_input.to_string(index=False))
print(f"Purchase probability: {probability:.1%}")


 Age TypeofContact  CityTier  DurationOfPitch Occupation Gender  NumberOfPersonVisiting  NumberOfFollowups ProductPitched  PreferredPropertyStar MaritalStatus  NumberOfTrips  Passport  PitchSatisfactionScore  OwnCar  NumberOfChildrenVisiting Designation  MonthlyIncome
35.0  Self Enquiry         2             12.0   Salaried Female                       2                3.0          Basic                    4.0        Single            2.0         1                       3       1                       0.0   Executive        25000.0
Purchase probability: 52.3%


In [9]:
print((PROJECT_ROOT / "deployment" / "requirements.txt").read_text())


pandas==2.2.3
numpy==2.3.5
scikit-learn==1.8.0
joblib==1.5.3
streamlit==1.45.1


### Streamlit Community Cloud deployment steps

1. Push the repository to a **public** GitHub repository on the `main` branch.
2. Wait for **Wellness Tourism MLOps Pipeline** to complete successfully in the Actions tab.
3. In Streamlit Community Cloud, choose the repository, branch `main`, and main file path `app.py`.
4. Confirm the public app loads and run one prediction.

The rubric names Streamlit Community Cloud. If your course portal separately insists on a Hugging Face Spaces URL, deploy this same `app.py` as a public Streamlit Space as an additional mirror and include that URL as well.


## 8. Business recommendations

1. **Use probability to rank leads, not as an automatic decision.** At the evaluated threshold, precision is 94.8%; a high-scoring customer is a strong candidate for a sales follow-up.
2. **Choose the campaign threshold with Operations.** The current setting is conservative: it produces only 5 false positives but misses 67 purchasers. A lower threshold may be appropriate if the cost of an additional call is low.
3. **Prioritize relevant conversation, not demographic exclusion.** Passport status, age, income, trip history, and pitch attributes are influential signals, but the marketing team should use them to personalize outreach and monitor results by segment for fairness.
4. **Monitor campaign outcomes and retrain deliberately.** Keep the model only if new data continues to meet the quality gate. Review precision, recall, and calibration after major changes in offer price, targeting strategy, or customer mix.


## 9. Output evaluation - complete after public deployment

The repository and app source are complete and the local pipeline has been run successfully. Public URLs and screenshots cannot be created without the student's GitHub and Streamlit accounts, so do **not** submit this section with placeholder text. After deployment, replace the four fields below and export this notebook to HTML again.

| Required evidence | Paste before final submission |
|---|---|
| Public GitHub repository URL | `https://github.com/<your-username>/<your-repository>` |
| Screenshot of repository structure | Insert a screenshot showing `data`, `src`, `models`, `reports`, `app.py`, and `.github/workflows/pipeline.yml` |
| Screenshot of completed Actions run | Insert a screenshot showing all three workflow jobs as successful |
| Public Streamlit app URL and screenshot | `https://<your-app>.streamlit.app` plus a screenshot of one completed prediction |

### Push command reference

```bash
cd visit_with_us_mlops
git init
git add .
git commit -m "Initial Wellness Tourism MLOps pipeline"
git branch -M main
git remote add origin https://github.com/<your-username>/<your-repository>.git
git push -u origin main
```

After adding the actual URLs and evidence, use **File -> Download -> HTML** in Jupyter/Colab, or run `pandoc Visit_with_Us_MLOps_Submission.ipynb --standalone --embed-resources -o Visit_with_Us_MLOps_Submission.html`.


## Final conclusion

The completed workflow converts a manual marketing-selection process into a traceable MLOps pipeline. It keeps raw data, preparation, experimentation, validation, deployment, and CI/CD controls connected so Visit with Us can refresh the model safely as customer behaviour evolves.
